# 04 · Compare — Base vs Fine-Tuned vs Reference Instruct

**Runs on: Local CPU**  
**Prerequisite**: Download `outputs/adapter/` from Colab after running notebook 03.

This notebook is the payoff. You'll run the same prompts through three models and see the difference fine-tuning makes:

| Model | Description |
|---|---|
| **Base** | `SmolLM2-135M` — no instruction tuning, raw language model |
| **Fine-tuned** | Base + your LoRA adapter trained in notebook 03 |
| **Reference** | `SmolLM2-135M-Instruct` — HF's own fine-tune of the same base |

In [ ]:
%pip install transformers peft accelerate

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

BASE_ID = "HuggingFaceTB/SmolLM2-135M"
REFERENCE_ID = "HuggingFaceTB/SmolLM2-135M-Instruct"
ADAPTER_PATH = "../outputs/adapter"  # relative to notebooks/ folder

DTYPE = torch.float32  # CPU-safe; use bfloat16 on GPU

## 1. Load All Three Models

In [ ]:
print("Loading base model...")
base_tokenizer = AutoTokenizer.from_pretrained(BASE_ID)
if base_tokenizer.pad_token is None:
    base_tokenizer.pad_token = base_tokenizer.eos_token
base_model = AutoModelForCausalLM.from_pretrained(BASE_ID, torch_dtype=DTYPE)
base_model.eval()
print("  Base model ready.")

In [ ]:
print("Loading fine-tuned model (base + your LoRA adapter)...")

import os
if not os.path.exists(ADAPTER_PATH):
    print(f"ERROR: Adapter not found at '{ADAPTER_PATH}'")
    print("Did you download outputs/adapter/ from Colab after notebook 03?")
else:
    # Load base again, then wrap with your LoRA adapter
    ft_base = AutoModelForCausalLM.from_pretrained(BASE_ID, torch_dtype=DTYPE)
    ft_model = PeftModel.from_pretrained(ft_base, ADAPTER_PATH)
    ft_model.eval()
    ft_tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH)
    if ft_tokenizer.pad_token is None:
        ft_tokenizer.pad_token = ft_tokenizer.eos_token
    print("  Fine-tuned model ready.")

In [ ]:
print("Loading reference Instruct model...")
ref_tokenizer = AutoTokenizer.from_pretrained(REFERENCE_ID)
if ref_tokenizer.pad_token is None:
    ref_tokenizer.pad_token = ref_tokenizer.eos_token
ref_model = AutoModelForCausalLM.from_pretrained(REFERENCE_ID, torch_dtype=DTYPE)
ref_model.eval()
print("  Reference model ready.")

## 2. Generate Helper

In [ ]:
CHATML = (
    "{% for message in messages %}"
    "{{'<|im_start|>' + message['role'] + '\\n' + message['content'] + '<|im_end|>' + '\\n'}}"
    "{% endfor %}"
    "{% if add_generation_prompt %}{{ '<|im_start|>assistant\\n' }}{% endif %}"
)

def respond(model, tokenizer, user_message, max_new_tokens=120, temperature=0.3):
    messages = [{"role": "user", "content": user_message}]
    prompt = tokenizer.apply_chat_template(
        messages, chat_template=CHATML, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_toks = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_toks, skip_special_tokens=True).strip()

## 3. Side-by-Side Comparison

Run the same instruction through all three models. Watch how the base model fails to follow instructions while both fine-tuned versions succeed.

In [ ]:
test_prompts = [
    "What is the capital of Japan? Answer in one word.",
    "Explain what a neural network is in two sentences.",
    "Write a haiku about winter snow.",
    "List three tips for sleeping better.",
    "What is 15% of 200?",
]

DIVIDER = "=" * 70

for i, prompt in enumerate(test_prompts, 1):
    print(f"\n{DIVIDER}")
    print(f"PROMPT {i}: {prompt}")
    print(DIVIDER)

    print("\n[BASE - no instruction tuning]")
    print(respond(base_model, base_tokenizer, prompt))

    if os.path.exists(ADAPTER_PATH):
        print("\n[YOUR FINE-TUNE - 2k samples, 1 epoch, LoRA r=8]")
        print(respond(ft_model, ft_tokenizer, prompt))

    print("\n[REFERENCE (SmolLM2-135M-Instruct) - HF's full training]")
    print(respond(ref_model, ref_tokenizer, prompt))

## 4. Reflection — What Did Fine-Tuning Actually Learn?

Run this cell to think through what changed:

In [ ]:
# Inspect what LoRA weights look like after training
if os.path.exists(ADAPTER_PATH):
    print("LoRA adapter weight norms (how much each layer changed):")
    for name, param in ft_model.named_parameters():
        if "lora_" in name and param.requires_grad:
            norm = param.norm().item()
            print(f"  {name:60s}  norm={norm:.4f}")

## 5. What the Numbers Mean

Your fine-tuned model was trained on only 2,000 samples for 1 epoch vs. HF's full training (the reference is trained on significantly more data and epochs). Expect:

- **Your model** should follow instructions in a structured way (vs. base which just continues text)
- **Your model** may be less fluent or accurate than the reference — that's expected
- **Reference** will generally be better — it's what professional training at HF scale produces

The gap between your model and the reference shows what **more data + more epochs + better hypertuning** can do.  
The gap between base and your model shows that **even 2k samples of LoRA SFT** teaches the model the format.

**Next**: [`05_push_to_hub.ipynb`](05_push_to_hub.ipynb) — publish your adapter to the HF Hub!